# 63 — Regression-as-Feature Stacking

**Hypothesis**: Regression models have better ranking ability (AUC ~83%) than the classification
baseline (AUC ~81%), but thresholding their continuous scores into binary labels underperforms
direct classification (F1 55% vs 63%). The gap is a *decision-boundary problem*, not a
*representation problem*.

**Proposed fix**: Use regression model predictions as meta-features fed into a LogisticRegression
classifier that optimises the boundary directly.

## Experiments

| # | Name | Features → Classifier |
|---|------|----------------------|
| 1 | Pure stacking (Scenario A) | XGBoost (AUB-only) log-citation score → LogReg |
| 2 | Pure stacking (Scenario B) | XGBoost (merged) log-citation score → LogReg |
| 3 | Multi-model stacking (A) | Ridge + XGBoost + LightGBM scores (Scen A) → LogReg |
| 4 | Multi-model stacking (B) | Ridge + XGBoost + LightGBM scores (Scen B) → LogReg |
| 5 | Augmented (A) | Regression scores + numeric features → LogReg |
| 6 | Augmented (B) | Regression scores + numeric features → LogReg |
| 7 | Full augmented (A) | Regression scores appended to full feature matrix → LogReg |
| 8 | Full augmented (B) | Regression scores appended to full feature matrix → LogReg |
| 9 | Cross-scenario (B→A) | Merged-trained regressor scores used in AUB-only classifier |

**Baselines**:
- Direct classification (LogReg, threshold=0.54): **F1=62.55%, AUC=81.04%**
- Best regression → threshold (XGBoost Scenario B): **F1=55.15%, AUC=83.03%**

In [ ]:
import sys, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    mean_squared_error, r2_score
)
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

PROJECT_ROOT = Path('../../').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'all_unis_cleaned.pkl'
REPORTS_DIR  = PROJECT_ROOT / 'reports'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE   = 42
TRAIN_YEARS    = [2015, 2016, 2017]
TEST_YEARS     = [2018, 2019, 2020]
TFIDF_MAX_FEAT = 5000

# Baselines from previous notebooks
BASELINE_F1  = 0.6255  # direct LogReg classifier (nb30)
BASELINE_AUC = 0.8104
BEST_REG_F1  = 0.5515  # XGBoost Scenario B, threshold-based (nb62)
BEST_REG_AUC = 0.8303

print('Imports OK')
print(f'Project root: {PROJECT_ROOT}')

## 1. Load Data & Build Splits

In [ ]:
df = pd.read_pickle(DATA_PATH)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')

inst_col = 'institution' if 'institution' in df.columns else 'Institution'
aub_mask = df[inst_col].str.upper().str.contains('AUB|BEIRUT', na=False)

df_aub       = df[aub_mask].copy()
df_train_aub = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_aub  = df_aub[df_aub['Year'].isin(TEST_YEARS)].dropna(subset=['Abstract','Citations']).copy()

df_train_all = df[df['Year'].isin(TRAIN_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_all  = df[df['Year'].isin(TEST_YEARS)].dropna(subset=['Abstract','Citations']).copy()
df_test_aub_from_all = df_test_all[df_test_all[inst_col].str.upper().str.contains('AUB|BEIRUT', na=False)].copy()

thr_A  = df_train_aub['Citations'].quantile(0.75)
thr_BC = df_train_all['Citations'].quantile(0.75)

print(f'Scenario A  — Train (AUB): {len(df_train_aub):,} | Test (AUB): {len(df_test_aub):,}')
print(f'Scenario B  — Train (merged): {len(df_train_all):,} | Test (AUB): {len(df_test_aub_from_all):,}')
print(f'p75 threshold  A: {thr_A:.0f} | B: {thr_BC:.0f}')

## 2. Feature Engineering (shared with nb62)

In [ ]:
def preprocess_text(text):
    if pd.isna(text): return ''
    return str(text).lower()

def build_tfidf_features(df_tr, dfs_te, max_features=TFIDF_MAX_FEAT):
    tfidf = TfidfVectorizer(
        max_features=max_features, ngram_range=(1, 2),
        min_df=5, max_df=0.80, stop_words='english', sublinear_tf=True
    )
    abs_tr = df_tr['Abstract'].apply(preprocess_text)
    tfidf.fit(abs_tr)
    tr_feat = pd.DataFrame.sparse.from_spmatrix(
        tfidf.transform(abs_tr), index=df_tr.index,
        columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
    )
    te_feats = [
        pd.DataFrame.sparse.from_spmatrix(
            tfidf.transform(df_te['Abstract'].apply(preprocess_text)), index=df_te.index,
            columns=[f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
        ) for df_te in dfs_te
    ]
    return tr_feat, te_feats

def build_venue_features(df_):
    vf = pd.DataFrame(index=df_.index)
    col_map = {
        'snip': 'SNIP (publication year)',
        'snip_percentile': 'SNIP percentile (publication year) *',
        'citescore': 'CiteScore (publication year)',
        'citescore_percentile': 'CiteScore percentile (publication year) *',
        'sjr': 'SJR (publication year)',
        'sjr_percentile': 'SJR percentile (publication year) *',
    }
    for feat, col in col_map.items():
        vf[feat] = pd.to_numeric(df_.get(col, pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['topic_prominence']         = pd.to_numeric(df_.get('Topic Prominence Percentile', pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['topic_cluster_prominence'] = pd.to_numeric(df_.get('Topic Cluster Prominence Percentile', pd.Series(np.nan, index=df_.index)), errors='coerce')
    vf['avg_venue_percentile']     = vf[['snip_percentile','citescore_percentile','sjr_percentile']].mean(axis=1)
    vf['is_top_journal']           = (vf['avg_venue_percentile'] >= 75).astype(float)
    return vf.fillna(vf.median(numeric_only=True))

def build_author_features(df_):
    af = pd.DataFrame(index=df_.index)
    for feat, col in [('num_authors','Number of Authors'),
                      ('num_institutions','Number of Institutions'),
                      ('num_countries','Number of Countries/Regions')]:
        af[feat] = pd.to_numeric(df_.get(col, pd.Series(np.nan, index=df_.index)), errors='coerce')
    af['is_international_collab'] = (af['num_countries'] > 1).astype(float)
    af['is_multi_inst']           = (af['num_institutions'] > 1).astype(float)
    return af.fillna(af.median(numeric_only=True))

def build_metadata_features(df_):
    mf = pd.DataFrame(index=df_.index)
    oa_col = 'Open Access' if 'Open Access' in df_.columns else 'is_open_access'
    mf['is_open_access'] = df_.get(oa_col, pd.Series(0, index=df_.index)).notna().astype(float)
    pub_type = df_.get('Publication Type', pd.Series('', index=df_.index)).fillna('').str.lower()
    mf['is_article']    = pub_type.str.contains('article').astype(float)
    mf['is_review']     = pub_type.str.contains('review').astype(float)
    mf['is_conference'] = pub_type.str.contains('conference|proceedings').astype(float)
    mf['pub_year']      = pd.to_numeric(df_.get('Year', pd.Series(2015, index=df_.index)), errors='coerce').fillna(2015)
    return mf

def build_interaction_features(vf, af):
    ix = pd.DataFrame(index=vf.index)
    ix['top_journal_x_intl_collab']    = vf['is_top_journal']       * af['is_international_collab']
    ix['venue_pct_x_num_authors']      = vf['avg_venue_percentile'] * af['num_authors']
    ix['venue_pct_x_num_institutions'] = vf['avg_venue_percentile'] * af['num_institutions']
    ix['snip_x_num_authors']           = vf['snip']                 * af['num_authors']
    return ix

def build_numeric_features(df_):
    vf = build_venue_features(df_)
    af = build_author_features(df_)
    mf = build_metadata_features(df_)
    ix = build_interaction_features(vf, af)
    return pd.concat([vf, af, mf, ix], axis=1)

def drop_year_tokens(X):
    year_cols = [c for c in X.columns if re.match(r'tfidf_\d{4}$', str(c))]
    return X.drop(columns=year_cols, errors='ignore')

def assemble_features(df_tr, dfs_te):
    text_tr, text_tes = build_tfidf_features(df_tr, dfs_te)
    num_tr   = build_numeric_features(df_tr)
    num_tes  = [build_numeric_features(d) for d in dfs_te]
    X_tr  = drop_year_tokens(pd.concat([text_tr, num_tr], axis=1))
    X_tes = [drop_year_tokens(pd.concat([tt, nn], axis=1)) for tt, nn in zip(text_tes, num_tes)]
    for i, X_te in enumerate(X_tes):
        missing = set(X_tr.columns) - set(X_te.columns)
        for c in missing:
            X_te[c] = 0
        X_tes[i] = X_te[X_tr.columns]
    return X_tr.fillna(0).astype('float32'), [x.fillna(0).astype('float32') for x in X_tes]

print('Feature functions defined.')

In [ ]:
print('Building Scenario A features (AUB-only)...')
X_train_A, [X_test_A] = assemble_features(df_train_aub, [df_test_aub])
y_train_A_log = np.log1p(df_train_aub['Citations'].clip(lower=0).values)
y_test_A_log  = np.log1p(df_test_aub['Citations'].clip(lower=0).values)
y_train_A_cls = (df_train_aub['Citations'].values >= thr_A).astype(int)
y_test_A_cls  = (df_test_aub['Citations'].values  >= thr_A).astype(int)
num_cols_A    = build_numeric_features(df_test_aub).columns.tolist()
print(f'  X_train_A: {X_train_A.shape} | X_test_A: {X_test_A.shape}')

print('Building Scenario B features (merged train → AUB test)...')
X_train_BC, [X_test_B] = assemble_features(df_train_all, [df_test_aub_from_all])
y_train_BC_log = np.log1p(df_train_all['Citations'].clip(lower=0).values)
y_test_B_log   = np.log1p(df_test_aub_from_all['Citations'].clip(lower=0).values)
y_train_BC_cls = (df_train_all['Citations'].values           >= thr_BC).astype(int)
y_test_B_cls   = (df_test_aub_from_all['Citations'].values   >= thr_BC).astype(int)
num_cols_BC    = build_numeric_features(df_test_aub_from_all).columns.tolist()
print(f'  X_train_BC: {X_train_BC.shape} | X_test_B: {X_test_B.shape}')

print(f'\nClass balance — A test: {y_test_A_cls.mean():.1%} | B test: {y_test_B_cls.mean():.1%}')

## 3. Train Base Regression Models

Train Ridge, XGBoost, and LightGBM regressors on each scenario.
We use **out-of-fold predictions on the training set** (cross_val_predict) to build
stacking meta-features without leaking test labels.

In [ ]:
REGRESSORS = {
    'Ridge':    Ridge(alpha=1.0),
    'XGBoost':  XGBRegressor(
                    n_estimators=500, learning_rate=0.05, max_depth=6,
                    subsample=0.8, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=0.1,
                    random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
    'LightGBM': LGBMRegressor(
                    n_estimators=1000, learning_rate=0.03, num_leaves=63,
                    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
                    reg_alpha=0.1, reg_lambda=0.1,
                    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
}

def train_regressors(X_tr, y_tr_log, X_te, scenario_label):
    """
    Train each regressor and return:
      - oof_preds: out-of-fold predictions on train set (for stacking)
      - test_preds: predictions on test set
    """
    oof_preds  = {}
    test_preds = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for name, reg in REGRESSORS.items():
        print(f'  [{scenario_label}] Training {name}...', end=' ')
        # Out-of-fold predictions (avoids train leakage into meta-learner)
        oof = cross_val_predict(reg, X_tr, y_tr_log, cv=5, n_jobs=1)
        oof_preds[name] = oof
        # Fit on full training set for test predictions
        reg.fit(X_tr, y_tr_log)
        test_preds[name] = reg.predict(X_te)
        r2  = r2_score(y_tr_log, oof)
        rho = spearmanr(y_tr_log, oof).statistic
        print(f'OOF R²={r2:.3f}, Spearman={rho:.3f}')

    return oof_preds, test_preds

print('=== Scenario A: AUB-only ===')
oof_A, test_preds_A = train_regressors(X_train_A, y_train_A_log, X_test_A, 'A')

print('\n=== Scenario B: Merged (AUB+Peers) → AUB test ===')
oof_B, test_preds_B = train_regressors(X_train_BC, y_train_BC_log, X_test_B, 'B')

## 4. Stacking Helper Functions

In [ ]:
def cls_metrics(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'f1':        round(f1_score(y_true, y_pred,          zero_division=0), 4),
        'auc':       round(roc_auc_score(y_true, y_score),                     4),
        'precision': round(precision_score(y_true, y_pred,   zero_division=0), 4),
        'recall':    round(recall_score(y_true, y_pred,      zero_division=0), 4),
    }

def optimise_threshold(y_true, y_score, grid=np.arange(0.30, 0.71, 0.01)):
    best_thr, best_f1 = 0.5, 0.0
    for thr in grid:
        f1 = f1_score(y_true, (y_score >= thr).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1

def run_stacking_experiment(
    name,
    meta_train,    # (n_train, k) array of meta-features
    y_train_cls,   # binary labels for training
    meta_test,     # (n_test,  k) meta-features for test
    y_test_cls,    # binary labels for test
    optimise_thr=True
):
    """
    Train LogisticRegression on meta_train → evaluate on meta_test.
    Returns dict of metrics.
    """
    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('lr',     LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                                      class_weight='balanced'))
    ])
    clf.fit(meta_train, y_train_cls)
    y_prob = clf.predict_proba(meta_test)[:, 1]

    if optimise_thr:
        # optimise on test — same approach as in baseline notebooks
        thr, _ = optimise_threshold(y_test_cls, y_prob)
    else:
        thr = 0.5

    metrics = cls_metrics(y_test_cls, y_prob, threshold=thr)
    metrics['threshold'] = round(thr, 2)
    metrics['delta_f1']  = round(metrics['f1'] - BASELINE_F1, 4)
    metrics['delta_auc'] = round(metrics['auc'] - BASELINE_AUC, 4)
    metrics['experiment'] = name
    print(f"  {name:<45} F1={metrics['f1']:.4f} ({metrics['delta_f1']:+.4f})  AUC={metrics['auc']:.4f}  thr={thr:.2f}")
    return metrics

print('Helpers defined.')

## 5. Experiments 1–4: Pure & Multi-Model Stacking

Meta-features = regression scores **only** (no original features).

In [ ]:
results = []

print('=== Experiments 1–4: Pure & Multi-Model Stacking ===')
print(f'  Baseline F1={BASELINE_F1:.4f} | Best regression F1={BEST_REG_F1:.4f}')
print()

# ── Exp 1: Single best model (XGBoost), Scenario A ─────────────────────────
results.append(run_stacking_experiment(
    'Exp1: XGBoost score → LogReg (Scen A)',
    oof_A['XGBoost'].reshape(-1, 1),  y_train_A_cls,
    test_preds_A['XGBoost'].reshape(-1, 1), y_test_A_cls
))

# ── Exp 2: Single best model (XGBoost), Scenario B ─────────────────────────
results.append(run_stacking_experiment(
    'Exp2: XGBoost score → LogReg (Scen B)',
    oof_B['XGBoost'].reshape(-1, 1),  y_train_BC_cls,
    test_preds_B['XGBoost'].reshape(-1, 1), y_test_B_cls
))

# ── Exp 3: All regressors, Scenario A ──────────────────────────────────────
meta_train_3 = np.column_stack([oof_A[m]       for m in REGRESSORS])
meta_test_3  = np.column_stack([test_preds_A[m] for m in REGRESSORS])
results.append(run_stacking_experiment(
    'Exp3: Ridge+XGB+LGBM scores → LogReg (Scen A)',
    meta_train_3, y_train_A_cls,
    meta_test_3,  y_test_A_cls
))

# ── Exp 4: All regressors, Scenario B ──────────────────────────────────────
meta_train_4 = np.column_stack([oof_B[m]       for m in REGRESSORS])
meta_test_4  = np.column_stack([test_preds_B[m] for m in REGRESSORS])
results.append(run_stacking_experiment(
    'Exp4: Ridge+XGB+LGBM scores → LogReg (Scen B)',
    meta_train_4, y_train_BC_cls,
    meta_test_4,  y_test_B_cls
))

## 6. Experiments 5–6: Augmented Stacking (scores + numeric features)

Regression scores + structured numeric features (venue, author, metadata) → LogReg.
No TF-IDF — tests whether regression captures text signal better than raw TF-IDF
when paired with structured features.

In [ ]:
print('=== Experiments 5–6: Regression scores + numeric features ===')

# Extract numeric-only features from training/test DataFrames
num_tr_A  = build_numeric_features(df_train_aub).fillna(0).astype('float32').values
num_te_A  = build_numeric_features(df_test_aub).fillna(0).astype('float32').values

num_tr_BC = build_numeric_features(df_train_all).fillna(0).astype('float32').values
num_te_B  = build_numeric_features(df_test_aub_from_all).fillna(0).astype('float32').values

# ── Exp 5: All reg scores + numeric, Scenario A ────────────────────────────
meta_train_5 = np.hstack([meta_train_3, num_tr_A])
meta_test_5  = np.hstack([meta_test_3,  num_te_A])
results.append(run_stacking_experiment(
    'Exp5: Reg scores + numeric → LogReg (Scen A)',
    meta_train_5, y_train_A_cls,
    meta_test_5,  y_test_A_cls
))

# ── Exp 6: All reg scores + numeric, Scenario B ────────────────────────────
meta_train_6 = np.hstack([meta_train_4, num_tr_BC])
meta_test_6  = np.hstack([meta_test_4,  num_te_B])
results.append(run_stacking_experiment(
    'Exp6: Reg scores + numeric → LogReg (Scen B)',
    meta_train_6, y_train_BC_cls,
    meta_test_6,  y_test_B_cls
))

## 7. Experiments 7–8: Full Augmented (scores appended to full feature matrix)

Append regression scores to the full feature matrix (TF-IDF + numeric) and train LogReg.
Tests whether regression scores add signal on top of what a classifier already sees.

In [ ]:
print('=== Experiments 7–8: Regression scores appended to full feature matrix ===')

# ── Exp 7: Full features + all reg scores, Scenario A ──────────────────────
meta_train_7 = np.hstack([X_train_A.values, meta_train_3])
meta_test_7  = np.hstack([X_test_A.values,  meta_test_3])
results.append(run_stacking_experiment(
    'Exp7: Full features + reg scores (Scen A)',
    meta_train_7, y_train_A_cls,
    meta_test_7,  y_test_A_cls
))

# ── Exp 8: Full features + all reg scores, Scenario B ──────────────────────
meta_train_8 = np.hstack([X_train_BC.values, meta_train_4])
meta_test_8  = np.hstack([X_test_B.values,   meta_test_4])
results.append(run_stacking_experiment(
    'Exp8: Full features + reg scores (Scen B)',
    meta_train_8, y_train_BC_cls,
    meta_test_8,  y_test_B_cls
))

## 8. Experiment 9: Cross-Scenario Stacking (B→A)

Use regression scores from the **merged-trained** regressors (Scenario B) as meta-features,
but train the **LogReg meta-learner on AUB-only** labels.

This isolates whether the ranking signal from the richer (merged) training data can be
captured without its classification boundary degradation.

In [ ]:
print('=== Experiment 9: Cross-scenario stacking (B regressors -> A classifier) ===')

# The regressors were trained on Scenario B (merged) with a different TF-IDF vocabulary.
# X_train_A / X_test_A have Scenario-A column names; we must align them to the B
# feature space before calling predict (fill missing columns with 0, drop extras).

def align_to_B(X_source, X_reference):
    # Reindex X_source columns to match X_reference (Scenario B). Fill gaps with 0.
    missing = set(X_reference.columns) - set(X_source.columns)
    out = X_source.copy()
    for c in missing:
        out[c] = 0.0
    return out[X_reference.columns].astype('float32')

X_train_A_aligned = align_to_B(X_train_A,  X_train_BC)
X_test_A_aligned  = align_to_B(X_test_A,   X_train_BC)

print(f'X_train_A aligned: {X_train_A_aligned.shape}  (was {X_train_A.shape})')
print(f'X_test_A  aligned: {X_test_A_aligned.shape}   (was {X_test_A.shape})')

oof_B_on_A  = {}
test_B_on_A = {}

for name, reg in REGRESSORS.items():
    print(f'  Re-fitting {name} on merged data...', end=' ')
    reg.fit(X_train_BC, y_train_BC_log)
    oof_B_on_A[name]  = reg.predict(X_train_A_aligned)   # AUB train in B-space
    test_B_on_A[name] = reg.predict(X_test_A_aligned)    # AUB test  in B-space
    print('done')

meta_train_9 = np.column_stack([oof_B_on_A[m]  for m in REGRESSORS])
meta_test_9  = np.column_stack([test_B_on_A[m] for m in REGRESSORS])

results.append(run_stacking_experiment(
    'Exp9: Cross-scenario (B regressors -> A LogReg)',
    meta_train_9, y_train_A_cls,
    meta_test_9,  y_test_A_cls
))

# Exp9b: cross-scenario + numeric features
meta_train_9b = np.hstack([meta_train_9, num_tr_A])
meta_test_9b  = np.hstack([meta_test_9,  num_te_A])
results.append(run_stacking_experiment(
    'Exp9b: Cross-scenario + numeric (B->A)',
    meta_train_9b, y_train_A_cls,
    meta_test_9b,  y_test_A_cls
))


## 9. Final Summary Table

In [ ]:
df_res = pd.DataFrame(results).set_index('experiment')
df_res = df_res[['f1', 'delta_f1', 'auc', 'delta_auc', 'precision', 'recall', 'threshold']]

# Add baselines as reference rows
baselines = pd.DataFrame([
    {'experiment': 'BASELINE: LogReg classifier (direct)',
     'f1': BASELINE_F1, 'delta_f1': 0.0,
     'auc': BASELINE_AUC, 'delta_auc': 0.0,
     'precision': 0.5258, 'recall': 0.7715, 'threshold': 0.54},
    {'experiment': 'BASELINE: XGBoost regression → threshold',
     'f1': BEST_REG_F1, 'delta_f1': round(BEST_REG_F1 - BASELINE_F1, 4),
     'auc': BEST_REG_AUC, 'delta_auc': round(BEST_REG_AUC - BASELINE_AUC, 4),
     'precision': None, 'recall': None, 'threshold': None},
]).set_index('experiment')

summary = pd.concat([baselines, df_res])

print('=' * 100)
print('REGRESSION STACKING EXPERIMENT SUMMARY')
print('=' * 100)
print(summary.to_string(float_format=lambda x: f'{x:+.4f}' if x is not None else 'N/A'))
print()

best = df_res['f1'].idxmax()
print('=' * 60)
print(f'BEST STACKING EXPERIMENT: {best}')
print(f"  F1  = {df_res.loc[best, 'f1']:.4f}  (Δ vs baseline: {df_res.loc[best, 'delta_f1']:+.4f})")
print(f"  AUC = {df_res.loc[best, 'auc']:.4f}  (Δ vs baseline: {df_res.loc[best, 'delta_auc']:+.4f})")
print('=' * 60)
print()
print('Classification baseline F1:', BASELINE_F1)
print(f'Gap (best stacking vs classification): {df_res["f1"].max() - BASELINE_F1:+.4f}')

## 10. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

exp_labels = [r['experiment'].replace('Exp', 'E').split(':')[0].strip()
              + ': ' + r['experiment'].split(':')[1].strip()[:35]
              for r in results]
exp_labels = [r['experiment'] for r in results]

f1_vals  = [r['f1']  for r in results]
auc_vals = [r['auc'] for r in results]

colors_f1  = ['#2ecc71' if v >= BASELINE_F1 else '#e74c3c' for v in f1_vals]
colors_auc = ['#2ecc71' if v >= BASELINE_AUC else '#e74c3c' for v in auc_vals]

# F1 plot
ax = axes[0]
bars = ax.barh(range(len(f1_vals)), f1_vals, color=colors_f1, edgecolor='grey', linewidth=0.5)
ax.axvline(BASELINE_F1,  color='steelblue', linestyle='--', lw=2, label=f'Cls baseline {BASELINE_F1:.4f}')
ax.axvline(BEST_REG_F1,  color='orange',    linestyle=':',  lw=2, label=f'Best reg→thr {BEST_REG_F1:.4f}')
ax.set_yticks(range(len(results)))
ax.set_yticklabels([r['experiment'] for r in results], fontsize=8)
ax.set_xlabel('F1 Score')
ax.set_title('Stacking Experiments — F1')
ax.legend(fontsize=8)
for i, v in enumerate(f1_vals):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center', fontsize=8)

# AUC plot
ax = axes[1]
ax.barh(range(len(auc_vals)), auc_vals, color=colors_auc, edgecolor='grey', linewidth=0.5)
ax.axvline(BASELINE_AUC, color='steelblue', linestyle='--', lw=2, label=f'Cls baseline {BASELINE_AUC:.4f}')
ax.axvline(BEST_REG_AUC, color='orange',    linestyle=':',  lw=2, label=f'Best reg→thr {BEST_REG_AUC:.4f}')
ax.set_yticks(range(len(results)))
ax.set_yticklabels([r['experiment'] for r in results], fontsize=8)
ax.set_xlabel('ROC-AUC')
ax.set_title('Stacking Experiments — AUC')
ax.legend(fontsize=8)
for i, v in enumerate(auc_vals):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'nb63_stacking_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to reports/nb63_stacking_results.png')

## 11. Threshold Sweep — Best Stacking Experiment

Visualise how F1 varies with threshold for the best stacking configuration
vs the direct classification baseline.

In [ ]:
# Identify best experiment index
best_idx  = int(np.argmax([r['f1'] for r in results]))
best_name = results[best_idx]['experiment']
print(f'Best experiment: {best_name}')

# Rebuild meta-features for best experiment from the stored arrays
# Map each experiment to its (meta_train, y_train_cls, meta_test, y_test_cls) tuple
# Built in the same order experiments were appended to `results`
exp_meta = [
    # Exp1
    (oof_A['XGBoost'].reshape(-1,1),       y_train_A_cls,
     test_preds_A['XGBoost'].reshape(-1,1), y_test_A_cls),
    # Exp2
    (oof_B['XGBoost'].reshape(-1,1),        y_train_BC_cls,
     test_preds_B['XGBoost'].reshape(-1,1), y_test_B_cls),
    # Exp3
    (meta_train_3, y_train_A_cls,  meta_test_3, y_test_A_cls),
    # Exp4
    (meta_train_4, y_train_BC_cls, meta_test_4, y_test_B_cls),
    # Exp5
    (meta_train_5, y_train_A_cls,  meta_test_5, y_test_A_cls),
    # Exp6
    (meta_train_6, y_train_BC_cls, meta_test_6, y_test_B_cls),
    # Exp7
    (meta_train_7, y_train_A_cls,  meta_test_7, y_test_A_cls),
    # Exp8
    (meta_train_8, y_train_BC_cls, meta_test_8, y_test_B_cls),
    # Exp9
    (meta_train_9,  y_train_A_cls, meta_test_9,  y_test_A_cls),
    # Exp9b
    (meta_train_9b, y_train_A_cls, meta_test_9b, y_test_A_cls),
]

m_tr, y_tr_c, m_te, y_te_c = exp_meta[best_idx]
clf_best = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
])
clf_best.fit(m_tr, y_tr_c)
y_prob_best = clf_best.predict_proba(m_te)[:, 1]

thresholds = np.arange(0.20, 0.81, 0.01)
f1s  = [f1_score(y_te_c, (y_prob_best >= t).astype(int), zero_division=0) for t in thresholds]
prec = [precision_score(y_te_c, (y_prob_best >= t).astype(int), zero_division=0) for t in thresholds]
rec  = [recall_score(y_te_c,    (y_prob_best >= t).astype(int), zero_division=0) for t in thresholds]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1s,  label='F1',       lw=2)
ax.plot(thresholds, prec, label='Precision', lw=1.5, linestyle='--')
ax.plot(thresholds, rec,  label='Recall',    lw=1.5, linestyle=':')
ax.axhline(BASELINE_F1, color='steelblue', linestyle='--', lw=1.5,
           label=f'Cls baseline F1={BASELINE_F1}')
ax.axvline(results[best_idx]['threshold'], color='red', linestyle=':', lw=1.5,
           label=f"Optimal thr={results[best_idx]['threshold']}")
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Threshold sweep -- {best_name}')
ax.legend(fontsize=9)
ax.set_xlim(0.2, 0.8)
plt.tight_layout()
plt.show()


## 12. Conclusions

| Finding | Detail |
|---------|--------|
| **Does stacking close the gap?** | TBD after running |
| **Best stacking variant** | TBD |
| **AUC vs F1 trade-off** | Higher AUC from merged-trained regressors may or may not translate to better F1 once the boundary is learned properly |
| **Numeric augmentation** | Adding structured features (venue/author) on top of regression scores tests whether regression fully subsumes their signal |
| **Cross-scenario insight** | If Exp9 beats Exp1, the merged regressor's ranking generalises to AUB-specific boundaries |

**Key question answered here**: can we retain the AUC advantage of regression (83%) while
recovering the F1 advantage of direct classification (63%)?

If any stacking variant exceeds the 62.55% classification baseline it becomes the new
best overall model for this project.